# The minimum Python for working with SQLite

## What does a Python program need in order to ask a database a question?

In the [SQL guide](https://dvirross.com/courses/intro-data-science/sql-in-db-browser) you typed queries into DB Browser. A program sends the same queries and receives the rows as Python values. This lesson goes through the Python that such a program uses, in the order in which a program uses it: values and text, tuples and lists, the calls that talk to the database, parameters, a function that runs a query, and a list that is built from query results.

The data is the bike-share table `daily_ridership` of the guide: 731 real days, one row for each day. This lesson uses six of its columns: `year`, `month`, `season`, `weather` (one of three labels), `working_day` (1 for a working day, 0 otherwise) and `total_rides` (the rides of that day).

Where a topic has a full lesson of its own, this lesson repeats the idea in one sentence and links to it: [values and collections](https://dvirross.com/courses/intro-data-science/python-values-and-collections), [loops and functions](https://dvirross.com/courses/intro-data-science/python-loops-and-functions) and [SQL from Python](https://dvirross.com/courses/intro-data-science/sql-from-python).

**How to work:** run the cells from top to bottom, because the cells share their variables. A cell that says `# Your turn` is empty: write your code there, run it, and run the cell under it, which checks your answer without showing the solution. The notebook with solutions has the solution, the common mistakes and the expected output under every task.

## Setup: get the database file

No account and no key are needed. The next cell downloads the course database from the public site and saves it next to the notebook (in Colab it appears in the **Files** panel). If the download fails, open the address below in a browser, download the file, and upload it to the **Files** panel of Colab.

`https://dvirross.com/materials/courses/intro-data-science/sql-practice.sqlite`

In [1]:
import os
import urllib.request

DB_URL = 'https://dvirross.com/materials/courses/intro-data-science/sql-practice.sqlite'
DB_PATH = 'sql-practice.sqlite'

if not os.path.exists(DB_PATH):
    # the site refuses the default user agent of Python, so the request names one
    request = urllib.request.Request(DB_URL, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(request) as response, open(DB_PATH, 'wb') as file:
        file.write(response.read())
print('Database file ready:', DB_PATH)

Database file ready: sql-practice.sqlite


The next cell defines the function `check(task_number)`. After you run the cell of a task, run the cell under it: `check` runs your cell again, collects what it prints, and compares it with the expected output. It tells you whether the output is right, and where it differs if not; it never shows the solution.

In [2]:
import ast, contextlib, hashlib, io

# The expected output of each task, as hashes of its lines (the solutions are not stored here).
EXPECTED = {int(k): v for k, v in {
    '3': [
        '4963af400ece'
    ],
    '4': [
        '7d827bbe5897'
    ],
    '5': [
        '14e4e0edb1cb'
    ],
    '6': [
        '2b855e705c53',
        'fa4ea1bc266c'
    ],
    '7': [
        '28af1299a0b7'
    ]
}.items()}


def check(task):
    """Run the last cell that ran (not a check) again and compare what it prints with the expected output of the task."""
    if task not in EXPECTED:
        print('There is no such task: ', task)
        return
    cells = [s for s in In[:-1] if s.strip() and not s.lstrip().startswith('check(')]   # In: the cells that ran, kept by IPython
    tree = ast.parse(cells[-1] if cells else '')
    last = tree.body.pop() if tree.body and isinstance(tree.body[-1], ast.Expr) else None   # a bare last expression counts as printed
    out = io.StringIO()
    try:
        with contextlib.redirect_stdout(out):
            exec(compile(tree, '<task>', 'exec'), globals())
            if last is not None:
                value = eval(compile(ast.Expression(last.value), '<task>', 'eval'), globals())
                if value is not None:
                    print(repr(value))
    except Exception as error:
        print('The cell raised an error:', type(error).__name__ + ':', error)
        return
    got = [hashlib.sha256(line.rstrip().encode()).hexdigest()[:12] for line in out.getvalue().rstrip().splitlines()]
    want = EXPECTED[task]
    if not got:
        print('Nothing was printed. Print the answer in the task cell, run it, and run `check` straight after it.')
    elif len(got) != len(want):
        print('The cell printed {got} line(s) and the task expects {want}. Re-read the task and the hint.'.format(got=len(got), want=len(want)))
    elif got != want:
        print('Line {k} is different from the expected output. Re-read the task and the hint.'.format(k=next(i for i in range(len(want)) if got[i] != want[i]) + 1))
    else:
        print('OK: the output is as expected.')


## Values, variables and `print`

A **[variable](https://dvirross.com/courses/intro-data-science/glossary#variable)** is a name that refers to a value, such as the number 985 or the text `'Mist or cloudy'`. The function `print` shows values on the screen: it puts a space between one value and the next, so it can show text and numbers together. A line that starts with `#` is a **comment**: Python ignores it, and it is there for the reader.

In [3]:
# one day of the table
rides = 985
weather = 'Mist or cloudy'
print('Weather:', weather, 'Rides:', rides)

Weather: Mist or cloudy Rides: 985


## Text and quotes

Text is written between quotes, and single and double quotes are the same. Text in **triple quotes** can continue over several lines. SQL is text for Python, and it contains single quotes around its own text values, so it is written either in double quotes or in triple quotes:

In [4]:
label = 'Light rain or snow'
one_line = "SELECT * FROM daily_ridership WHERE year = 2012"
many_lines = """
    SELECT year, month
    FROM daily_ridership
    WHERE weather = 'Mist or cloudy'
"""
print(label)
print(one_line)

Light rain or snow
SELECT * FROM daily_ridership WHERE year = 2012


## Tuples, lists and indexing

A table in a database is rows under fixed columns. A query result arrives in Python as a list of **[tuples](https://dvirross.com/courses/intro-data-science/glossary#tuple)**: a tuple is written with round brackets and holds a fixed number of values in order, one for each column of the `SELECT`, and it cannot be changed by accident. A **[list](https://dvirross.com/courses/intro-data-science/glossary#list)** is written with square brackets and holds many items, here many rows. An **[index](https://dvirross.com/courses/intro-data-science/glossary#index)** is the position of an item in square brackets, and counting starts at 0, so `row[0]` is the first value.

In [5]:
row = (2011, 1, 985)
rows = [(2011, 1, 985), (2011, 1, 801), (2011, 1, 1349)]
print(row[0], row[2])
print(rows[1])
print(rows[1][2])

2011 985
(2011, 1, 801)
801


The last line takes the second row of the list and then the third value of that row. See [tuples and lists](https://dvirross.com/courses/intro-data-science/python-values-and-collections#tuples-and-lists-several-values-in-order) for the full lesson.

## Connect, cursor, execute, fetch

Four steps take rows from a database into Python. Each step is one line. The line `import sqlite3` makes the module `sqlite3`, which comes with Python, available.

**Connect.** The function `sqlite3.connect` opens the database file whose name you give it and returns a **[connection](https://dvirross.com/courses/intro-data-science/glossary#connection)**, an object that stands for the open database.

In [6]:
import sqlite3

conn = sqlite3.connect('sql-practice.sqlite')

**Create a cursor.** The method `cursor` of the connection creates a **[cursor](https://dvirross.com/courses/intro-data-science/glossary#cursor)**, an object that sends SQL and holds the answer until you take it.

In [7]:
c = conn.cursor()

**Execute a query.** The method `execute` of the cursor sends one query. It returns the cursor itself, so what you see is the cursor object, and the rows of the answer wait in it.

In [8]:
c.execute("""
    SELECT year, month, weather, total_rides
    FROM daily_ridership
    LIMIT 2
""")

**Fetch the results.** The method `fetchall` takes all the rows of the answer, as a list of tuples.

In [9]:
results = c.fetchall()
print(results)

[(2011, 1, 'Mist or cloudy', 985), (2011, 1, 'Mist or cloudy', 801)]


The method `fetchmany` takes the next rows, as many as the number you give it. The method `fetchone` takes the next row alone, as a tuple. A query can select 731 rows while `fetchmany(3)` takes three of them. Because `execute` returns the cursor, a fetch can follow it in the same line. The values of a row are in the order of the `SELECT`: the first selected column is index 0.

In [10]:
three = c.execute(
    "SELECT year, month, total_rides FROM daily_ridership"
).fetchmany(3)
print(three)
row = c.execute(
    "SELECT month, total_rides FROM daily_ridership LIMIT 1"
).fetchone()
print(row)
print('Month:', row[0])
print('Rides:', row[1])

[(2011, 1, 985), (2011, 1, 801), (2011, 1, 1349)]
(1, 985)
Month: 1
Rides: 985


## Summary functions in a query

The summary functions of the [SQL guide](https://dvirross.com/courses/intro-data-science/sql-in-db-browser#summarize-values) work the same inside a program. `SUM(total_rides)` adds the rides of the chosen days, while `COUNT(*)` counts the days; `COUNT(weather)` counts the rows that have a label and `COUNT(DISTINCT weather)` counts the different labels. A summary query returns one row, so `fetchone()` is the right fetch, and the row is a tuple even when it holds a single value.

In [11]:
result = c.execute("""
    SELECT SUM(total_rides), COUNT(*)
    FROM daily_ridership
    WHERE year = 2011 AND working_day = 0
""").fetchone()
print('Rides:', result[0], 'Days:', result[1])

Rides: 386839 Days: 115


The query needs `SUM` because each row is a day with many rides: `COUNT(*)` would return the number of days, 115, and not the rides.

### Task 1 (worked example): days and labels of 2012

How many days does 2012 have in the table, and how many different weather labels do they have? Compare `COUNT(weather)` with `COUNT(DISTINCT weather)`: what does each count? Decide before you run the code.

In [12]:
result = c.execute("""
    SELECT COUNT(weather), COUNT(DISTINCT weather)
    FROM daily_ridership
    WHERE year = 2012
""").fetchone()
print('Days:', result[0], 'Different weather labels:', result[1])

Days: 366 Different weather labels: 3


The year 2012 has 366 days (a leap year), every day has a label, and the labels are three in all. The row is a tuple with two values.

A query with `DISTINCT` returns one row for each value, and when it selects one column the rows are tuples with one value. Python writes such a tuple with a comma, `('Mist or cloudy',)`. `LIKE '%cloudy%'` matches a text that contains `cloudy` anywhere.

### Task 2 (worked example): labels that contain cloudy

How many different weather labels contain the text `cloudy`? Then list those labels.

In [13]:
result = c.execute("""
    SELECT COUNT(DISTINCT weather)
    FROM daily_ridership
    WHERE weather LIKE '%cloudy%'
""").fetchone()
print('Labels that contain cloudy:', result[0])
labels = c.execute("""
    SELECT DISTINCT weather
    FROM daily_ridership
    WHERE weather LIKE '%cloudy%'
    LIMIT 10
""").fetchall()
print(labels)

Labels that contain cloudy: 2
[('Mist or cloudy',), ('Clear or partly cloudy',)]


The count is `result[0]`, and the list holds a tuple with one value for each label.

### Task 3: rides on working days

**Your turn.** Use the cursor `c` to get the total number of rides on the working days of 2012. Print the text `Total rides:` and then the number, in one `print`.

In [14]:
# Your turn

**Expected output (the numbers are hidden)**

```text
Total rides: <number>
```

In [15]:
check(3)

> **Hint**
>
> Write the SQL as in DB Browser: one summary function, and a `WHERE` with two conditions joined by `AND`. The result is one row, so take it with `fetchone` and take the number out of the tuple with an index.

### Groups and the largest total

`GROUP BY` puts the rows with the same value together and `SUM` adds each group. `ORDER BY ... DESC` sorts the groups from the largest, and `LIMIT 1` keeps the first. Here they find the season with the most rides in 2011:

In [16]:
best = c.execute("""
    SELECT season, SUM(total_rides)
    FROM daily_ridership
    WHERE year = 2011
    GROUP BY season
    ORDER BY SUM(total_rides) DESC
    LIMIT 1
""").fetchone()
print('Season:', best[0], 'Rides:', best[1])

Season: Summer Rides: 419650


### Task 4: the busiest month

**Your turn.** Which month of the year had the most rides in the whole table? The rides of the same month in 2011 and in 2012 are added together. Print the text `Busiest month:` and then the tuple of the month number and its total rides, in one `print`.

In [17]:
# Your turn

**Expected output (the numbers are hidden)**

```text
Busiest month: (<number>, <number>)
```

In [18]:
check(4)

> **Hint**
>
> Group the rows by `month`, so that the same month of the two years is one group. Sort the groups by the sum in descending order and keep one. The row is a tuple of the month and the sum, and it is what the `print` shows.

## Values from Python variables: parameters

A program usually asks about values that it holds in variables. The values are passed as **parameters**: write a question mark `?` in the SQL where a value belongs, and give the values separately, in a tuple, as the second argument of `execute`. The question marks are filled in order. Python can create several variables in one line, as in the first line of the box below: the names on the left receive the values on the right, in order (this is [unpacking](https://dvirross.com/courses/intro-data-science/glossary#unpacking)).

In [19]:
weather, year, working_day = 'Mist or cloudy', 2011, 0
query = """
    SELECT SUM(total_rides)
    FROM daily_ridership
    WHERE weather = ? AND year = ? AND working_day = ?
"""
values = (weather, year, working_day)
result = c.execute(query, values).fetchone()
print('Rides:', result[0])

Rides: 102244


**Why not put the values into the text with an f-string?**

An [f-string](https://dvirross.com/courses/intro-data-science/glossary#f-string) pastes a value into the SQL text, and a value that contains SQL changes the query. A season that arrives as `Summer' OR '1'='1` closes the quote and adds a condition that is always true, so the query below counts all 731 days. This is [SQL injection](https://dvirross.com/courses/intro-data-science/glossary#sql-injection). With `?` the value is only data, and the same text matches no season:

In [20]:
season = "Summer' OR '1'='1"
pasted = f"""
    SELECT COUNT(*) FROM daily_ridership
    WHERE season = '{season}'
"""
print(c.execute(pasted).fetchone())
safe = """
    SELECT COUNT(*) FROM daily_ridership
    WHERE season = ?
"""
print(c.execute(safe, (season,)).fetchone())

(731,)
(0,)


[SQL from Python](https://dvirross.com/courses/intro-data-science/sql-from-python#passing-values-parameters) shows this in full.

### Task 5: values from variables

**Your turn.** Store the weather `'Clear or partly cloudy'`, the year 2012 and the working-day value 1 in three variables. Use them as three `?` parameters to get the total rides of those days, and print the text `Total rides:` and the number.

In [21]:
# Your turn

**Expected output (the numbers are hidden)**

```text
Total rides: <number>
```

In [22]:
check(5)

> **Hint**
>
> The SQL has the three conditions `weather = ?`, `year = ?` and `working_day = ?`, joined by `AND`. The tuple of values has the variables in the same order as the question marks.

## Functions that query

A **function** gives a name to a piece of work. The line `def get_name(parameter):` defines it, the indented lines below are its body, and `return` sends a value back to the caller. A query function takes the values it needs as parameters and passes them on as the `?` values. See [loops and functions](https://dvirross.com/courses/intro-data-science/python-loops-and-functions#functions-a-name-for-a-piece-of-work) for the full lesson.

A summary query over no rows returns a missing value: `SUM` of nothing is `NULL`, and Python shows `NULL` as **[`None`](https://dvirross.com/courses/intro-data-science/glossary#none)**, the value that means no value. A function that returns a number should return 0 in that case. A **[conditional expression](https://dvirross.com/courses/intro-data-science/glossary#conditional-expression)**, `a if condition else b`, is `a` when the condition is true and `b` otherwise, and `is not None` is true when a value exists:

In [23]:
print(
    c.execute("""
        SELECT SUM(total_rides)
        FROM daily_ridership
        WHERE season = 'Summer' AND year = 2013
    """).fetchone()
)

def get_season_rides(season, year):
    query = """
        SELECT SUM(total_rides)
        FROM daily_ridership
        WHERE season = ? AND year = ?
    """
    row = c.execute(query, (season, year)).fetchone()
    return row[0] if row[0] is not None else 0

print('Summer 2011:', get_season_rides('Summer', 2011))
print('Summer 2013:', get_season_rides('Summer', 2013))

(None,)
Summer 2011: 419650
Summer 2013: 0


The function uses the cursor `c` that was created above. The data covers the years 2011 and 2012, so the year 2013 has no rows.

### Task 6: a function that queries

**Your turn.** Write a function `get_total_rides(year, month)` that returns the total rides of the given year and month (a whole number from 1 to 12), and 0 when no row matches. Print the text `July 2012:` and the result for July 2012, then the text `January 2013:` and the result for January 2013, each on its own line.

In [24]:
# Your turn

**Expected output (the numbers are hidden)**

```text
July 2012: <number>
January 2013: 0
```

In [25]:
check(6)

> **Hint**
>
> Inside the function: a query with two `?` parameters, `fetchone`, and a conditional expression that returns 0 when the first value of the row is `None`.

## A list from query results, and `max` with a key

A **[list comprehension](https://dvirross.com/courses/intro-data-science/glossary#list-comprehension)** builds a list in one line: `[expression for item in rows]` gives the expression for every item. The function `max` returns the largest item of a list. When the items are tuples, `key=` says what to compare: it receives a function, and `lambda pair: pair[1]` is a **[lambda](https://dvirross.com/courses/intro-data-science/glossary#lambda)**, a one-line function without a name that returns the second value of a tuple. Then `max` returns the whole tuple whose second value is the largest.

The example below collects `(season, rides)` tuples for 2011, with the function from the previous section. The call `fetchall` completes before the loop starts, so the same cursor can run the queries inside it. Each row of the seasons is a tuple with one value, so `season[0]` is the text:

In [26]:
season_totals = [
    (season[0], get_season_rides(season[0], 2011))
    for season in c.execute(
        "SELECT DISTINCT season FROM daily_ridership"
    ).fetchall()
]
print(season_totals)
print(max(season_totals, key=lambda pair: pair[1]))

[('Winter', 150000), ('Spring', 347316), ('Summer', 419650), ('Fall', 326137)]
('Summer', 419650)


### Task 7: a list and `max`

**Your turn.** Write a function with this name and these parameters:

```text
get_rides_by_weather_year_month(weather, year, month)
```

It returns the total rides on the days of that weather in that year and month, and 0 when no row matches. Use it to build a list of `(month, rides)` tuples for the weather `'Clear or partly cloudy'` in 2012, with one tuple for every different month in the table (the table also has the months of 2011, so select the months of 2012). Find the tuple with the most rides with `max` and print it.

In [27]:
# Your turn

**Expected output (the numbers are hidden)**

```text
(<number>, <number>)
```

In [28]:
check(7)

> **Hint**
>
> The function is like `get_total_rides` with one more condition, `weather = ?`. The list comprehension runs over a query that selects the different months of 2012, and `month[0]` is the number of the month.

## Questions to talk about

- Which part of the code changes when the weather, the year or the month come from variables?
- What could go wrong if a value were pasted into the SQL text with an f-string?
- Where did a query result become an ordinary Python value in your code?

## Closing the connection

The method `close` of the connection closes the database when the work is done. Closing a connection does not save changes (a program that changes data calls `commit` first, see [SQL from Python](https://dvirross.com/courses/intro-data-science/sql-from-python#changing-data-and-saving-it)); our queries only read.

Run the cell below as the last step: it releases the database file.

In [29]:
conn.close()

## Keep in mind

- A program asks a database in four steps: `connect`, `cursor`, `execute`, `fetch`. `fetchall` returns a list of tuples, `fetchone` one tuple, and `fetchmany(n)` the next `n` tuples.
- The values of a row are in the order of the `SELECT`, and an index takes one of them: `row[0]`.
- Pass values with `?` and a tuple of values, never by pasting text into the SQL.
- A summary query over no rows gives `None`; a conditional expression, as in `get_total_rides`, turns it into 0.
- `max` with `key=lambda pair: pair[1]` returns the tuple with the largest second value.